In [15]:
import requests
from bs4 import BeautifulSoup
import csv
import re

url = 'https://wuzzuf.net/search/jobs?a=spbg&q=data%20engineer'

with open('Data_Engineer_Jobs.csv', 'w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerow(['index', 'Job_Name', 'Company_Name', 'Location', 'Job_Type', 'Experience_Level', 'Posting_Date'])

    global_idx = 1 

    def scrape_page(soup):
        global global_idx
        result = soup.find('div', class_='css-9i2afk')
        if not result:
            return
        findJobs = result.find_all('div', class_='css-pkv5jc')
        for i in findJobs:
            Job_Name = i.find('h2', class_='css-193uk2c').text.strip()
            Company_Name = i.find('a', class_='css-ipsyv7').text.strip().rstrip(' -')
            Location = i.find('span', class_='css-16x61xq').text.strip()
            Job_Type = i.find('span', class_='css-uc9rga eoyjyou0').text.strip()
            full_text = i.get_text()
            years_match = re.search(r'(\d+\+?\s*Yrs?\s*of\s*Exp)', full_text, re.IGNORECASE)
            Experience_Level = years_match.group(1).strip() if years_match else 'N/A'
            try:
                Posting_Date = i.find('div', class_='css-eg55jf').text.strip()
            except AttributeError:
                Posting_Date = 'Unknown'
            writer.writerow([global_idx, Job_Name, Company_Name, Location, Job_Type, Experience_Level, Posting_Date])
            global_idx += 1
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    scrape_page(soup)
    for page_num in range(29):
        page_url = f'https://wuzzuf.net/search/jobs?q=data%20engineer&start={page_num}&a=spbg'
        response = requests.get(page_url)  
        soup = BeautifulSoup(response.content, 'html.parser')
        scrape_page(soup)